<a href="https://colab.research.google.com/github/Pushkarsinghs/indian_stock-analysis/blob/main/00_master_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ══════════════════════════════════════════════════════
# 00 — MASTER UPDATE PIPELINE
# Generates all 10 Power BI files in one run
# Run every day after 3:30 PM IST
# ══════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

!pip install yfinance ta pyarrow prophet pyportfolioopt \
            textblob feedparser scipy -q

import pandas as pd
import numpy as np
import yfinance as yf
import ta
import os
import time
import warnings
from datetime import datetime
import pytz
warnings.filterwarnings("ignore")

BASE = '/content/drive/MyDrive/indian_stock_analysis'
IST  = pytz.timezone("Asia/Kolkata")
now  = datetime.now(IST)

# Make sure all folders exist
for folder in ["data/raw","data/processed","data/output"]:
    os.makedirs(f"{BASE}/{folder}", exist_ok=True)

print("=" * 60)
print(f"  🚀 NIFTY 50 COMPLETE DAILY PIPELINE")
print(f"  🕒 {now.strftime('%d %b %Y %H:%M:%S')} IST")
print(f"  📁 Output: {BASE}/data/output/")
print("=" * 60)

# ── Stock Lists ──────────────────────────────────────
NIFTY_50 = [
    "RELIANCE.NS","TCS.NS","HDFCBANK.NS","INFY.NS","ICICIBANK.NS",
    "HINDUNILVR.NS","ITC.NS","SBIN.NS","BHARTIARTL.NS","KOTAKBANK.NS",
    "LT.NS","AXISBANK.NS","ASIANPAINT.NS","MARUTI.NS","SUNPHARMA.NS",
    "TITAN.NS","ULTRACEMCO.NS","BAJFINANCE.NS","WIPRO.NS","ONGC.NS",
    "NTPC.NS","POWERGRID.NS","TECHM.NS","HCLTECH.NS","JSWSTEEL.NS",
    "TATASTEEL.NS","TATAMOTORS.NS","NESTLEIND.NS","DRREDDY.NS","DIVISLAB.NS",
    "CIPLA.NS","COALINDIA.NS","BPCL.NS","GRASIM.NS","ADANIENT.NS",
    "ADANIPORTS.NS","BAJAJFINSV.NS","BAJAJ-AUTO.NS","HEROMOTOCO.NS","EICHERMOT.NS",
    "BRITANNIA.NS","HINDALCO.NS","UPL.NS","SBILIFE.NS","HDFCLIFE.NS",
    "APOLLOHOSP.NS","TATACONSUM.NS","INDUSINDBK.NS","M&M.NS","LTF.NS"
]

SECTOR_MAP = {
    "RELIANCE.NS":"Energy","TCS.NS":"IT","HDFCBANK.NS":"Banking",
    "INFY.NS":"IT","ICICIBANK.NS":"Banking","HINDUNILVR.NS":"FMCG",
    "ITC.NS":"FMCG","SBIN.NS":"Banking","BHARTIARTL.NS":"Telecom",
    "KOTAKBANK.NS":"Banking","LT.NS":"Infrastructure","AXISBANK.NS":"Banking",
    "ASIANPAINT.NS":"Paints","MARUTI.NS":"Auto","SUNPHARMA.NS":"Pharma",
    "TITAN.NS":"Consumer","ULTRACEMCO.NS":"Cement","BAJFINANCE.NS":"NBFC",
    "WIPRO.NS":"IT","ONGC.NS":"Energy","NTPC.NS":"Power",
    "POWERGRID.NS":"Power","TECHM.NS":"IT","HCLTECH.NS":"IT",
    "JSWSTEEL.NS":"Steel","TATASTEEL.NS":"Steel","TATAMOTORS.NS":"Auto",
    "NESTLEIND.NS":"FMCG","DRREDDY.NS":"Pharma","DIVISLAB.NS":"Pharma",
    "CIPLA.NS":"Pharma","COALINDIA.NS":"Mining","BPCL.NS":"Energy",
    "GRASIM.NS":"Cement","ADANIENT.NS":"Conglomerate","ADANIPORTS.NS":"Ports",
    "BAJAJFINSV.NS":"NBFC","BAJAJ-AUTO.NS":"Auto","HEROMOTOCO.NS":"Auto",
    "EICHERMOT.NS":"Auto","BRITANNIA.NS":"FMCG","HINDALCO.NS":"Metals",
    "UPL.NS":"Agrochemicals","SBILIFE.NS":"Insurance","HDFCLIFE.NS":"Insurance",
    "APOLLOHOSP.NS":"Healthcare","TATACONSUM.NS":"FMCG",
    "INDUSINDBK.NS":"Banking","M&M.NS":"Auto","LTF.NS":"NBFC"
}

COMPANY_NAMES = {
    "RELIANCE.NS":"Reliance Industries","TCS.NS":"TCS Tata Consultancy",
    "HDFCBANK.NS":"HDFC Bank","INFY.NS":"Infosys","ICICIBANK.NS":"ICICI Bank",
    "HINDUNILVR.NS":"Hindustan Unilever","ITC.NS":"ITC Limited",
    "SBIN.NS":"State Bank India SBI","BHARTIARTL.NS":"Bharti Airtel",
    "KOTAKBANK.NS":"Kotak Mahindra Bank","LT.NS":"Larsen Toubro",
    "AXISBANK.NS":"Axis Bank","ASIANPAINT.NS":"Asian Paints",
    "MARUTI.NS":"Maruti Suzuki","SUNPHARMA.NS":"Sun Pharma",
    "TITAN.NS":"Titan Company","ULTRACEMCO.NS":"UltraTech Cement",
    "BAJFINANCE.NS":"Bajaj Finance","WIPRO.NS":"Wipro",
    "ONGC.NS":"ONGC Oil Gas","NTPC.NS":"NTPC Power",
    "POWERGRID.NS":"Power Grid India","TECHM.NS":"Tech Mahindra",
    "HCLTECH.NS":"HCL Technologies","JSWSTEEL.NS":"JSW Steel",
    "TATASTEEL.NS":"Tata Steel","TATAMOTORS.NS":"Tata Motors",
    "NESTLEIND.NS":"Nestle India","DRREDDY.NS":"Dr Reddys",
    "DIVISLAB.NS":"Divis Laboratories","CIPLA.NS":"Cipla Pharma",
    "COALINDIA.NS":"Coal India","BPCL.NS":"BPCL Petroleum",
    "GRASIM.NS":"Grasim Industries","ADANIENT.NS":"Adani Enterprises",
    "ADANIPORTS.NS":"Adani Ports","BAJAJFINSV.NS":"Bajaj Finserv",
    "BAJAJ-AUTO.NS":"Bajaj Auto","HEROMOTOCO.NS":"Hero MotoCorp",
    "EICHERMOT.NS":"Eicher Motors","BRITANNIA.NS":"Britannia Industries",
    "HINDALCO.NS":"Hindalco Aluminium","UPL.NS":"UPL Agrochemicals",
    "SBILIFE.NS":"SBI Life Insurance","HDFCLIFE.NS":"HDFC Life Insurance",
    "APOLLOHOSP.NS":"Apollo Hospitals","TATACONSUM.NS":"Tata Consumer",
    "INDUSINDBK.NS":"IndusInd Bank","M&M.NS":"Mahindra","LTF.NS":"LT Finance"
}

# ════════════════════════════════════════════════════
# FILE 1 & 2 — RAW PRICE DATA
# Generates: nifty50_for_powerbi.csv
#            nifty50_latest_day.csv
# ════════════════════════════════════════════════════
print("\n📥 [1/6] Fetching live NSE data...\n")

# ── Ticker alternatives for Yahoo Finance quirks ──
TICKER_ALTERNATIVES = {
    "TATAMOTORS.NS": ["TATAMOTORS.NS", "TATAMTRDVR.NS"],
    "M&M.NS":        ["M&M.NS",        "MM.NS"],
    "BAJAJ-AUTO.NS": ["BAJAJ-AUTO.NS", "BAJAJ_AUTO.NS"],
    "LTF.NS":        ["LTF.NS",        "L&TFH.NS"]
}

def fetch_with_fallback(ticker, period="1y"):
    """Try multiple ticker formats if primary fails"""
    alternatives = TICKER_ALTERNATIVES.get(ticker, [ticker])
    for alt_ticker in alternatives:
        try:
            df = yf.Ticker(alt_ticker).history(period=period)
            if not df.empty:
                df["Ticker"] = ticker
                df["Sector"] = SECTOR_MAP.get(ticker, "Unknown")
                df.reset_index(inplace=True)
                return df
        except Exception:
            continue
    return None

# ── Fetch all stocks with fallback ──
all_data = []
failed   = []

for i, ticker in enumerate(NIFTY_50, 1):
    df = fetch_with_fallback(ticker)
    if df is not None and not df.empty:
        all_data.append(df)
        print(f"  ✅ [{i:02d}/{len(NIFTY_50)}] {ticker} — {len(df)} rows")
    else:
        failed.append(ticker)
        print(f"  ⚠️  [{i:02d}/{len(NIFTY_50)}] {ticker} — skipped")

print(f"\n  ✅ Fetched: {len(all_data)} stocks")
if failed:
    print(f"  ⚠️  Skipped: {failed}")
    print(f"  💡 Skipped stocks won't affect the rest of the pipeline")

raw_df = pd.concat(all_data, ignore_index=True)
raw_df["Date"] = pd.to_datetime(raw_df["Date"]).dt.tz_localize(None)
raw_df = raw_df[["Date","Ticker","Sector","Open","High","Low","Close","Volume"]]
raw_df.drop_duplicates(subset=["Date","Ticker"], inplace=True)
raw_df.dropna(subset=["Close"], inplace=True)
raw_df.sort_values(["Ticker","Date"], inplace=True)
raw_df["Daily_Return"]     = raw_df.groupby("Ticker")["Close"].pct_change().round(4)
raw_df["Price_Change"]     = (raw_df["Close"] - raw_df["Open"]).round(2)
raw_df["Price_Change_Pct"] = ((raw_df["Price_Change"] / raw_df["Open"]) * 100).round(2)
raw_df.reset_index(drop=True, inplace=True)

# Save parquet for analysis
raw_df.to_parquet(f"{BASE}/data/processed/nifty50_master.parquet", index=False)

# FILE 1
raw_df.to_csv(f"{BASE}/data/output/nifty50_for_powerbi.csv", index=False)
print(f"\n  💾 FILE 1 saved → nifty50_for_powerbi.csv ({len(raw_df):,} rows)")

# FILE 2
latest_day = raw_df[raw_df["Date"] == raw_df["Date"].max()].copy()
latest_day.to_csv(f"{BASE}/data/output/nifty50_latest_day.csv", index=False)
print(f"  💾 FILE 2 saved → nifty50_latest_day.csv ({len(latest_day)} rows)")
print(f"\n✅ [1/6] Complete")

# ════════════════════════════════════════════════════
# FILE 3 & 4 — TECHNICAL ANALYSIS
# Generates: nifty50_technical_powerbi.csv
#            latest_signals.csv
# ════════════════════════════════════════════════════
print("\n📈 [2/6] Computing technical indicators...\n")

tech_results = []
for i, ticker in enumerate(raw_df["Ticker"].unique(), 1):
    try:
        s = raw_df[raw_df["Ticker"] == ticker].copy().sort_values("Date")

        # Trend
        s["SMA_20"]      = ta.trend.SMAIndicator(s["Close"], window=20).sma_indicator()
        s["SMA_50"]      = ta.trend.SMAIndicator(s["Close"], window=50).sma_indicator()
        s["SMA_200"]     = ta.trend.SMAIndicator(s["Close"], window=200).sma_indicator()
        s["EMA_20"]      = ta.trend.EMAIndicator(s["Close"], window=20).ema_indicator()
        s["EMA_26"]      = ta.trend.EMAIndicator(s["Close"], window=26).ema_indicator()
        macd             = ta.trend.MACD(s["Close"])
        s["MACD"]        = macd.macd()
        s["MACD_Signal"] = macd.macd_signal()
        s["MACD_Hist"]   = macd.macd_diff()
        s["ADX"]         = ta.trend.ADXIndicator(
                               s["High"], s["Low"], s["Close"]
                           ).adx()

        # Momentum
        s["RSI"]         = ta.momentum.RSIIndicator(s["Close"], window=14).rsi()
        stoch            = ta.momentum.StochasticOscillator(
                               s["High"], s["Low"], s["Close"]
                           )
        s["Stoch_K"]     = stoch.stoch()
        s["Stoch_D"]     = stoch.stoch_signal()
        s["Williams_R"]  = ta.momentum.WilliamsRIndicator(
                               s["High"], s["Low"], s["Close"]
                           ).williams_r()

        # Volatility
        bb               = ta.volatility.BollingerBands(s["Close"])
        s["BB_Upper"]    = bb.bollinger_hband()
        s["BB_Middle"]   = bb.bollinger_mavg()
        s["BB_Lower"]    = bb.bollinger_lband()
        s["BB_Width"]    = bb.bollinger_wband()
        s["ATR"]         = ta.volatility.AverageTrueRange(
                               s["High"], s["Low"], s["Close"]
                           ).average_true_range()

        # Volume
        s["OBV"]           = ta.volume.OnBalanceVolumeIndicator(
                                 s["Close"], s["Volume"]
                             ).on_balance_volume()
        s["Volume_SMA_20"] = s["Volume"].rolling(20).mean()

        # Signal Score
        s["Signal_Score"] = 0
        s.loc[s["RSI"] < 30,                           "Signal_Score"] += 2
        s.loc[s["RSI"] > 70,                           "Signal_Score"] -= 2
        s.loc[s["MACD"] > s["MACD_Signal"],            "Signal_Score"] += 1
        s.loc[s["MACD"] < s["MACD_Signal"],            "Signal_Score"] -= 1
        s.loc[s["Close"] > s["SMA_50"],                "Signal_Score"] += 1
        s.loc[s["Close"] < s["SMA_50"],                "Signal_Score"] -= 1
        s.loc[s["Close"] > s["SMA_200"],               "Signal_Score"] += 1
        s.loc[s["Close"] < s["SMA_200"],               "Signal_Score"] -= 1
        s.loc[s["Close"] < s["BB_Lower"],              "Signal_Score"] += 1
        s.loc[s["Close"] > s["BB_Upper"],              "Signal_Score"] -= 1

        s["Signal"]     = "Neutral"
        s.loc[s["Signal_Score"] >= 3,  "Signal"] = "Strong Buy"
        s.loc[s["Signal_Score"] == 2,  "Signal"] = "Buy"
        s.loc[s["Signal_Score"] == 1,  "Signal"] = "Weak Buy"
        s.loc[s["Signal_Score"] == -1, "Signal"] = "Weak Sell"
        s.loc[s["Signal_Score"] == -2, "Signal"] = "Sell"
        s.loc[s["Signal_Score"] <= -3, "Signal"] = "Strong Sell"

        s["RSI_Signal"] = "Neutral"
        s.loc[s["RSI"] < 30, "RSI_Signal"] = "Oversold"
        s.loc[s["RSI"] > 70, "RSI_Signal"] = "Overbought"

        tech_results.append(s)
        print(f"  ✅ [{i:02d}/{len(raw_df['Ticker'].unique())}] {ticker}")

    except Exception as e:
        print(f"  ❌ {ticker} — {e}")

tech_df = pd.concat(tech_results, ignore_index=True)
tech_df.to_parquet(f"{BASE}/data/processed/nifty50_technical.parquet", index=False)

# FILE 3
tech_df.to_csv(f"{BASE}/data/output/nifty50_technical_powerbi.csv", index=False)
print(f"\n  💾 FILE 3 saved → nifty50_technical_powerbi.csv ({len(tech_df):,} rows)")

# FILE 4
latest_signals = tech_df.groupby("Ticker").last().reset_index()
latest_signals.to_csv(f"{BASE}/data/output/latest_signals.csv", index=False)
print(f"  💾 FILE 4 saved → latest_signals.csv ({len(latest_signals)} rows)")
print(f"\n✅ [2/6] Complete")

# ════════════════════════════════════════════════════
# FILE 5 — FUNDAMENTAL ANALYSIS
# Generates: nifty50_fundamentals_powerbi.csv
# ════════════════════════════════════════════════════
print("\n📊 [3/6] Fetching fundamental data...\n")

fund_rows = []
for i, ticker in enumerate(NIFTY_50, 1):
    try:
        info = yf.Ticker(ticker).info
        fund_rows.append({
            "Ticker":               ticker,
            "Company":              info.get("longName",           "N/A"),
            "Sector":               SECTOR_MAP.get(ticker,         "Unknown"),
            "Industry":             info.get("industry",           "N/A"),
            "Market_Cap_Cr":        round(info.get("marketCap",0) / 1e7, 2),
            "PE_Ratio":             info.get("trailingPE",         None),
            "Forward_PE":           info.get("forwardPE",          None),
            "PB_Ratio":             info.get("priceToBook",        None),
            "PS_Ratio":             info.get("priceToSalesTrailing12Months", None),
            "EV_EBITDA":            info.get("enterpriseToEbitda", None),
            "ROE_Pct":              round(info.get("returnOnEquity",  0)*100, 2) if info.get("returnOnEquity")  else None,
            "ROA_Pct":              round(info.get("returnOnAssets",  0)*100, 2) if info.get("returnOnAssets")  else None,
            "Profit_Margin_Pct":    round(info.get("profitMargins",   0)*100, 2) if info.get("profitMargins")   else None,
            "Operating_Margin_Pct": round(info.get("operatingMargins",0)*100, 2) if info.get("operatingMargins") else None,
            "Revenue_Growth_Pct":   round(info.get("revenueGrowth",  0)*100, 2) if info.get("revenueGrowth")   else None,
            "Earnings_Growth_Pct":  round(info.get("earningsGrowth", 0)*100, 2) if info.get("earningsGrowth")  else None,
            "Debt_To_Equity":       info.get("debtToEquity",       None),
            "Current_Ratio":        info.get("currentRatio",       None),
            "Quick_Ratio":          info.get("quickRatio",         None),
            "EPS":                  info.get("trailingEps",        None),
            "Book_Value":           info.get("bookValue",          None),
            "Dividend_Yield_Pct":   round(info.get("dividendYield",0)*100, 2) if info.get("dividendYield") else 0,
            "Current_Price":        info.get("currentPrice",       info.get("regularMarketPrice", None)),
            "52W_High":             info.get("fiftyTwoWeekHigh",   None),
            "52W_Low":              info.get("fiftyTwoWeekLow",    None),
        })
        print(f"  ✅ [{i:02d}/{len(NIFTY_50)}] {ticker}")
    except Exception as e:
        print(f"  ❌ [{i:02d}] {ticker} — {e}")

fund_df = pd.DataFrame(fund_rows)

# Scoring
fund_df["Fund_Score"] = 0
fund_df.loc[fund_df["PE_Ratio"].between(0,15),   "Fund_Score"] += 15
fund_df.loc[fund_df["PE_Ratio"].between(15,25),  "Fund_Score"] += 10
fund_df.loc[fund_df["PE_Ratio"].between(25,40),  "Fund_Score"] += 5
fund_df.loc[fund_df["PB_Ratio"].between(0,1),    "Fund_Score"] += 10
fund_df.loc[fund_df["PB_Ratio"].between(1,3),    "Fund_Score"] += 7
fund_df.loc[fund_df["ROE_Pct"] > 20,             "Fund_Score"] += 20
fund_df.loc[fund_df["ROE_Pct"].between(15,20),   "Fund_Score"] += 15
fund_df.loc[fund_df["ROE_Pct"].between(10,15),   "Fund_Score"] += 10
fund_df.loc[fund_df["Profit_Margin_Pct"] > 20,   "Fund_Score"] += 15
fund_df.loc[fund_df["Profit_Margin_Pct"].between(10,20), "Fund_Score"] += 10
fund_df.loc[fund_df["Revenue_Growth_Pct"] > 20,  "Fund_Score"] += 15
fund_df.loc[fund_df["Revenue_Growth_Pct"].between(10,20),"Fund_Score"] += 10
fund_df.loc[fund_df["Debt_To_Equity"] < 0.5,     "Fund_Score"] += 15
fund_df.loc[fund_df["Debt_To_Equity"].between(0.5,1.0),"Fund_Score"] += 10
fund_df.loc[fund_df["Current_Ratio"] > 2,        "Fund_Score"] += 10
fund_df.loc[fund_df["Current_Ratio"].between(1.5,2),"Fund_Score"] += 7

fund_df["Fund_Grade"] = "F"
fund_df.loc[fund_df["Fund_Score"] >= 70,          "Fund_Grade"] = "A"
fund_df.loc[fund_df["Fund_Score"].between(55,69), "Fund_Grade"] = "B"
fund_df.loc[fund_df["Fund_Score"].between(40,54), "Fund_Grade"] = "C"
fund_df.loc[fund_df["Fund_Score"].between(25,39), "Fund_Grade"] = "D"

fund_df.to_parquet(f"{BASE}/data/processed/nifty50_fundamentals.parquet", index=False)

# FILE 5
fund_df.to_csv(f"{BASE}/data/output/nifty50_fundamentals_powerbi.csv", index=False)
print(f"\n  💾 FILE 5 saved → nifty50_fundamentals_powerbi.csv ({len(fund_df)} rows)")
print(f"\n✅ [3/6] Complete")

# ════════════════════════════════════════════════════
# FILE 6 — SENTIMENT ANALYSIS
# Generates: nifty50_sentiment_powerbi.csv
# ════════════════════════════════════════════════════
print("\n💬 [4/6] Fetching news sentiment...\n")

from textblob import TextBlob
import feedparser

BULLISH_KW = ["profit","growth","record","surge","rally","gain","strong",
              "beat","upgrade","buy","positive","robust","rise","jump",
              "soar","outperform","expansion","dividend","order","launch"]
BEARISH_KW = ["loss","fall","decline","drop","weak","miss","downgrade",
              "sell","negative","cut","layoff","fraud","penalty","probe",
              "investigate","debt","default","crash","plunge","warning"]

sentiment_rows = []
for i, (ticker, company) in enumerate(COMPANY_NAMES.items(), 1):
    try:
        query = company.replace(" ","+") + "+NSE+stock"
        url   = f"https://news.google.com/rss/search?q={query}&hl=en-IN&gl=IN&ceid=IN:en"
        feed  = feedparser.parse(url)
        headlines = []
        for entry in feed.entries[:8]:
            title = entry.title.rsplit(" - ",1)[0] if " - " in entry.title else entry.title
            headlines.append(title)

        if not headlines:
            sentiment_rows.append({
                "Ticker":ticker,"Company":company,"Avg_Polarity":0,
                "Sentiment_Label":"Neutral","Bullish_Count":0,
                "Bearish_Count":0,"Total_Articles":0,"Sentiment_Score":50
            })
            continue

        pols = []
        for h in headlines:
            base = TextBlob(h).sentiment.polarity
            hl   = h.lower()
            boost = sum(0.1 for w in BULLISH_KW if w in hl)
            boost -= sum(0.1 for w in BEARISH_KW if w in hl)
            pols.append(max(-1.0, min(1.0, base + boost)))

        avg    = round(np.mean(pols), 4)
        label  = ("Very Positive" if avg >= 0.3 else
                  "Positive"      if avg >= 0.1 else
                  "Very Negative" if avg <= -0.3 else
                  "Negative"      if avg <= -0.1 else "Neutral")

        sentiment_rows.append({
            "Ticker":          ticker,
            "Company":         company,
            "Avg_Polarity":    avg,
            "Sentiment_Label": label,
            "Bullish_Count":   sum(1 for p in pols if p > 0.1),
            "Bearish_Count":   sum(1 for p in pols if p < -0.1),
            "Total_Articles":  len(headlines),
            "Sentiment_Score": round((avg + 1) / 2 * 100, 1)
        })
        print(f"  ✅ [{i:02d}/{len(COMPANY_NAMES)}] {ticker} — {label}")
        time.sleep(0.3)

    except Exception as e:
        print(f"  ❌ {ticker} — {e}")

sentiment_df = pd.DataFrame(sentiment_rows)
sentiment_df.to_parquet(f"{BASE}/data/processed/nifty50_sentiment.parquet", index=False)

# FILE 6
sentiment_df.to_csv(f"{BASE}/data/output/nifty50_sentiment_powerbi.csv", index=False)
print(f"\n  💾 FILE 6 saved → nifty50_sentiment_powerbi.csv ({len(sentiment_df)} rows)")
print(f"\n✅ [4/6] Complete")

# ════════════════════════════════════════════════════
# FILE 7 & 8 — FORECASTING
# Generates: nifty50_forecasts_powerbi.csv
#            forecast_summary_powerbi.csv
# ════════════════════════════════════════════════════
print("\n🔮 [5/6] Running Prophet forecasts...\n")

from prophet import Prophet

all_forecasts   = []
summary_rows    = []

for i, ticker in enumerate(tech_df["Ticker"].unique(), 1):
    try:
        stock_data = tech_df[tech_df["Ticker"] == ticker][["Date","Close"]].copy()
        stock_data.columns = ["ds","y"]
        stock_data["ds"]   = pd.to_datetime(stock_data["ds"])
        stock_data         = stock_data.dropna().drop_duplicates("ds").sort_values("ds")

        if len(stock_data) < 60:
            print(f"  ⚠️  [{i:02d}] {ticker} — not enough data")
            continue

        model = Prophet(
            daily_seasonality  = False,
            weekly_seasonality = True,
            yearly_seasonality = True,
            changepoint_prior_scale = 0.05,
            interval_width          = 0.80
        )
        model.fit(stock_data)
        future   = model.make_future_dataframe(periods=30, freq="B")
        forecast = model.predict(future)

        fc_df = forecast[["ds","yhat","yhat_lower","yhat_upper"]].copy()
        fc_df.columns = ["Date","Forecast","Lower_CI","Upper_CI"]
        fc_df["Ticker"] = ticker
        for col in ["Forecast","Lower_CI","Upper_CI"]:
            fc_df[col] = fc_df[col].round(2)

        all_forecasts.append(fc_df)

        latest_price  = stock_data["y"].iloc[-1]
        predicted_30d = fc_df["Forecast"].iloc[-1]
        change_pct    = round((predicted_30d - latest_price) / latest_price * 100, 2)

        summary_rows.append({
            "Ticker":         ticker,
            "Current_Price":  round(latest_price, 2),
            "Forecast_30d":   round(predicted_30d, 2),
            "Lower_CI":       round(fc_df["Lower_CI"].iloc[-1], 2),
            "Upper_CI":       round(fc_df["Upper_CI"].iloc[-1], 2),
            "Expected_Change": change_pct,
            "Direction":      "Bullish 📈" if change_pct > 0 else "Bearish 📉",
            "Forecast_Date":  datetime.today().strftime("%Y-%m-%d")
        })

        direction = "📈" if change_pct > 0 else "📉"
        print(f"  ✅ [{i:02d}/{len(tech_df['Ticker'].unique())}] "
              f"{ticker} ₹{latest_price:.0f}→₹{predicted_30d:.0f} "
              f"({change_pct:+.1f}%) {direction}")

    except Exception as e:
        print(f"  ❌ {ticker} — {e}")

forecast_df = pd.concat(all_forecasts, ignore_index=True)
forecast_df.to_parquet(f"{BASE}/data/processed/nifty50_forecasts.parquet", index=False)

# FILE 7
forecast_df.to_csv(f"{BASE}/data/output/nifty50_forecasts_powerbi.csv", index=False)
print(f"\n  💾 FILE 7 saved → nifty50_forecasts_powerbi.csv ({len(forecast_df):,} rows)")

# FILE 8
summary_df = pd.DataFrame(summary_rows).sort_values("Expected_Change", ascending=False)
summary_df.to_csv(f"{BASE}/data/output/forecast_summary_powerbi.csv", index=False)
print(f"  💾 FILE 8 saved → forecast_summary_powerbi.csv ({len(summary_df)} rows)")
print(f"\n✅ [5/6] Complete")

# ════════════════════════════════════════════════════
# FILE 9 & 10 — RISK METRICS + PORTFOLIO
# Generates: nifty50_risk_metrics_powerbi.csv
#            portfolio_allocation_powerbi.csv
#            portfolio_performance_powerbi.csv
# ════════════════════════════════════════════════════
print("\n💼 [6/6] Computing risk metrics and portfolio optimization...\n")

from pypfopt import EfficientFrontier, risk_models, expected_returns, DiscreteAllocation

# Rebuild prices pivot
prices = tech_df.pivot_table(index="Date", columns="Ticker", values="Close").sort_index()
prices = prices.dropna(thresh=int(len(prices)*0.9), axis=1)
prices = prices.ffill().bfill()
prices = prices.loc[:, prices.notna().sum() >= 200]
returns_daily = prices.pct_change().dropna()

# Risk Metrics
risk_rows      = []
market_returns = returns_daily.mean(axis=1)
market_var     = market_returns.var()
trading_days   = 252
risk_free      = 0.065

for ticker in returns_daily.columns:
    try:
        r, mkt = returns_daily[ticker].dropna().align(market_returns, join="inner")
        r      = r.dropna()
        if len(r) < 30:
            continue

        ann_return   = round((1 + r.mean()) ** trading_days - 1, 4)
        ann_vol      = round(r.std() * np.sqrt(trading_days), 4)
        sharpe       = round((ann_return - risk_free) / ann_vol, 3) if ann_vol > 0 else 0
        cumulative   = (1 + r).cumprod()
        max_drawdown = round(((cumulative - cumulative.cummax()) / cumulative.cummax()).min() * 100, 2)
        threshold    = np.percentile(r, 5)
        var_95       = round(threshold * 100, 3)
        cvar_95      = round(r[r <= threshold].mean() * 100, 3) if len(r[r <= threshold]) > 0 else var_95
        cov_matrix   = np.cov(r.values, mkt.values)
        beta         = round(cov_matrix[0][1] / market_var, 3) if market_var > 0 else 1.0
        curr_price   = prices[ticker].iloc[-1]
        high_52w     = prices[ticker].tail(252).max()
        low_52w      = prices[ticker].tail(252).min()

        risk_rows.append({
            "Ticker":              ticker,
            "Ann_Return_Pct":      round(ann_return * 100, 2),
            "Ann_Volatility_Pct":  round(ann_vol * 100, 2),
            "Sharpe_Ratio":        sharpe,
            "Max_Drawdown_Pct":    max_drawdown,
            "VaR_95_Pct":          var_95,
            "CVaR_95_Pct":         cvar_95,
            "Beta":                beta,
            "Current_Price":       round(curr_price, 2),
            "52W_High":            round(high_52w, 2),
            "52W_Low":             round(low_52w, 2),
            "From_52W_High_Pct":   round((curr_price - high_52w) / high_52w * 100, 2)
        })
        print(f"  ✅ {ticker} | Sharpe: {sharpe:.2f} | Return: {ann_return*100:.1f}%")

    except Exception as e:
        print(f"  ❌ {ticker} — {e}")

risk_df = pd.DataFrame(risk_rows).sort_values("Sharpe_Ratio", ascending=False)
risk_df.to_parquet(f"{BASE}/data/processed/nifty50_risk_metrics.parquet", index=False)

# FILE 9
risk_df.to_csv(f"{BASE}/data/output/nifty50_risk_metrics_powerbi.csv", index=False)
print(f"\n  💾 FILE 9 saved → nifty50_risk_metrics_powerbi.csv ({len(risk_df)} rows)")

# Portfolio Optimization
print("\n  ⚙️  Running portfolio optimization...")
try:
    mu = expected_returns.mean_historical_return(prices, frequency=252)
    S  = risk_models.CovarianceShrinkage(prices).ledoit_wolf()

    perf_rows = []

    # Max Sharpe
    ef1 = EfficientFrontier(mu, S)
    ef1.add_constraint(lambda w: w >= 0.0)
    ef1.add_constraint(lambda w: w <= 0.15)
    ef1.max_sharpe(risk_free_rate=0.065)
    w_sharpe   = ef1.clean_weights(cutoff=0.005)
    p_sharpe   = ef1.portfolio_performance(verbose=False, risk_free_rate=0.065)
    perf_rows.append({
        "Portfolio":"Max Sharpe",
        "Ann_Return": round(p_sharpe[0]*100,2),
        "Ann_Risk":   round(p_sharpe[1]*100,2),
        "Sharpe_Ratio": round(p_sharpe[2],3)
    })

    # Min Volatility
    ef2 = EfficientFrontier(mu, S)
    ef2.add_constraint(lambda w: w >= 0.0)
    ef2.add_constraint(lambda w: w <= 0.15)
    ef2.min_volatility()
    w_minvol = ef2.clean_weights(cutoff=0.005)
    p_minvol = ef2.portfolio_performance(verbose=False, risk_free_rate=0.065)
    perf_rows.append({
        "Portfolio":"Min Volatility",
        "Ann_Return": round(p_minvol[0]*100,2),
        "Ann_Risk":   round(p_minvol[1]*100,2),
        "Sharpe_Ratio": round(p_minvol[2],3)
    })

    # Max Return
    try:
        ef3 = EfficientFrontier(mu, S)
        ef3.add_constraint(lambda w: w >= 0.0)
        ef3.add_constraint(lambda w: w <= 0.15)
        ef3.efficient_return(target_return=float(mu.max() * 0.90))
        w_maxret = ef3.clean_weights(cutoff=0.005)
        p_maxret = ef3.portfolio_performance(verbose=False, risk_free_rate=0.065)
        perf_rows.append({
            "Portfolio":"Max Return",
            "Ann_Return": round(p_maxret[0]*100,2),
            "Ann_Risk":   round(p_maxret[1]*100,2),
            "Sharpe_Ratio": round(p_maxret[2],3)
        })
    except:
        perf_rows.append(perf_rows[0].copy())
        perf_rows[-1]["Portfolio"] = "Max Return"

    # Portfolio performance table
    perf_df = pd.DataFrame(perf_rows)
    perf_df.to_csv(f"{BASE}/data/output/portfolio_performance_powerbi.csv", index=False)
    print(f"  💾 portfolio_performance_powerbi.csv saved")

    # Discrete Allocation
    non_zero = {k:v for k,v in w_sharpe.items() if v > 0.005}
    latest_prices_dict = {
        k: float(prices[k].iloc[-1])
        for k in non_zero if k in prices.columns
    }
    total_w      = sum(non_zero[k] for k in latest_prices_dict)
    weights_norm = {k: non_zero[k]/total_w for k in latest_prices_dict}

    INVESTMENT   = 100000
    max_price    = max(latest_prices_dict.values())
    min_required = max_price * len(latest_prices_dict)
    if INVESTMENT < min_required:
        INVESTMENT = int((min_required * 1.5) / 10000 + 1) * 10000
        print(f"  ⚠️  Auto-scaled investment to ₹{INVESTMENT:,}")

    try:
        da = DiscreteAllocation(
            weights_norm,
            pd.Series(latest_prices_dict),
            total_portfolio_value=INVESTMENT
        )
        allocation, leftover = da.greedy_portfolio()
    except:
        da = DiscreteAllocation(
            weights_norm,
            pd.Series(latest_prices_dict),
            total_portfolio_value=INVESTMENT
        )
        allocation, leftover = da.lp_portfolio()

    alloc_rows = []
    total_inv  = 0
    for ticker, shares in allocation.items():
        if shares <= 0 or ticker not in latest_prices_dict:
            continue
        price = latest_prices_dict[ticker]
        value = round(shares * price, 2)
        total_inv += value
        alloc_rows.append({
            "Ticker":     ticker,
            "Company":    ticker.replace(".NS",""),
            "Shares":     shares,
            "Price":      round(price,2),
            "Value_INR":  value,
            "Weight_Pct": round(weights_norm.get(ticker,0)*100,2)
        })

    alloc_df = pd.DataFrame(alloc_rows).sort_values("Value_INR", ascending=False)

    # FILE 10
    alloc_df.to_csv(f"{BASE}/data/output/portfolio_allocation_powerbi.csv", index=False)
    print(f"  💾 FILE 10 saved → portfolio_allocation_powerbi.csv ({len(alloc_df)} stocks)")
    print(f"  💰 Total invested: ₹{total_inv:,.0f} | Leftover: ₹{leftover:,.0f}")

except Exception as e:
    print(f"  ❌ Portfolio optimization failed: {e}")

print(f"\n✅ [6/6] Complete")

# ════════════════════════════════════════════════════
# FINAL CONFIRMATION — ALL 10 FILES
# ════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"  🎉 PIPELINE COMPLETE — ALL FILES SUMMARY")
print("=" * 60)

expected_files = [
    "nifty50_for_powerbi.csv",
    "nifty50_latest_day.csv",
    "nifty50_technical_powerbi.csv",
    "latest_signals.csv",
    "nifty50_fundamentals_powerbi.csv",
    "nifty50_sentiment_powerbi.csv",
    "nifty50_forecasts_powerbi.csv",
    "forecast_summary_powerbi.csv",
    "nifty50_risk_metrics_powerbi.csv",
    "portfolio_allocation_powerbi.csv",
    "portfolio_performance_powerbi.csv"
]

output_dir = f"{BASE}/data/output"
print(f"\n  {'#':<4} {'File':<45} {'Size':>8} {'Status':>8}")
print(f"  {'-'*70}")

all_good = True
for idx, fname in enumerate(expected_files, 1):
    fpath = f"{output_dir}/{fname}"
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        status = "✅ OK" if size > 1 else "⚠️  EMPTY"
        if size < 1:
            all_good = False
    else:
        size   = 0
        status = "❌ MISSING"
        all_good = False
    print(f"  {idx:<4} {fname:<45} {size:>6.1f}KB {status:>8}")

print(f"\n  {'='*60}")
if all_good:
    print(f"  ✅ ALL 10 FILES GENERATED SUCCESSFULLY")
else:
    print(f"  ⚠️  SOME FILES MISSING — CHECK ERRORS ABOVE")

now_end = datetime.now(IST)
print(f"  🕒 Completed: {now_end.strftime('%d %b %Y %H:%M:%S')} IST")
print(f"  ⏱️  Total time: {(now_end-now).seconds//60}m {(now_end-now).seconds%60}s")
print(f"\n  👉 NEXT STEP: Click REFRESH in Power BI Desktop")
print(f"  {'='*60}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  🚀 NIFTY 50 COMPLETE DAILY PIPELINE
  🕒 29 Jun 2026 10:52:20 IST
  📁 Output: /content/drive/MyDrive/indian_stock_analysis/data/output/

📥 [1/6] Fetching live NSE data...

  ✅ [01/50] RELIANCE.NS — 248 rows
  ✅ [02/50] TCS.NS — 248 rows
  ✅ [03/50] HDFCBANK.NS — 248 rows
  ✅ [04/50] INFY.NS — 248 rows
  ✅ [05/50] ICICIBANK.NS — 248 rows
  ✅ [06/50] HINDUNILVR.NS — 248 rows
  ✅ [07/50] ITC.NS — 248 rows
  ✅ [08/50] SBIN.NS — 248 rows
  ✅ [09/50] BHARTIARTL.NS — 248 rows
  ✅ [10/50] KOTAKBANK.NS — 248 rows
  ✅ [11/50] LT.NS — 248 rows
  ✅ [12/50] AXISBANK.NS — 248 rows
  ✅ [13/50] ASIANPAINT.NS — 248 rows
  ✅ [14/50] MARUTI.NS — 248 rows
  ✅ [15/50] SUNPHARMA.NS — 248 rows
  ✅ [16/50] TITAN.NS — 248 rows
  ✅ [17/50] ULTRACEMCO.NS — 248 rows
  ✅ [18/50] BAJFINANCE.NS — 248 rows
  ✅ [19/50] WIPRO.NS — 248 rows
  ✅ [20/50] ONGC.NS — 248 rows
  ✅ [21/50] NTPC.NS — 

ERROR:yfinance:$TATAMOTORS.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
ERROR:yfinance:$TATAMTRDVR.NS: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")


  ⚠️  [27/50] TATAMOTORS.NS — skipped
  ✅ [28/50] NESTLEIND.NS — 248 rows
  ✅ [29/50] DRREDDY.NS — 248 rows
  ✅ [30/50] DIVISLAB.NS — 247 rows
  ✅ [31/50] CIPLA.NS — 248 rows
  ✅ [32/50] COALINDIA.NS — 248 rows
  ✅ [33/50] BPCL.NS — 248 rows
  ✅ [34/50] GRASIM.NS — 248 rows
  ✅ [35/50] ADANIENT.NS — 248 rows
  ✅ [36/50] ADANIPORTS.NS — 248 rows
  ✅ [37/50] BAJAJFINSV.NS — 248 rows
  ✅ [38/50] BAJAJ-AUTO.NS — 248 rows
  ✅ [39/50] HEROMOTOCO.NS — 248 rows
  ✅ [40/50] EICHERMOT.NS — 248 rows
  ✅ [41/50] BRITANNIA.NS — 248 rows
  ✅ [42/50] HINDALCO.NS — 248 rows
  ✅ [43/50] UPL.NS — 248 rows
  ✅ [44/50] SBILIFE.NS — 248 rows
  ✅ [45/50] HDFCLIFE.NS — 248 rows
  ✅ [46/50] APOLLOHOSP.NS — 248 rows
  ✅ [47/50] TATACONSUM.NS — 248 rows
  ✅ [48/50] INDUSINDBK.NS — 248 rows
  ✅ [49/50] M&M.NS — 248 rows
  ✅ [50/50] LTF.NS — 248 rows

  ✅ Fetched: 49 stocks
  ⚠️  Skipped: ['TATAMOTORS.NS']
  💡 Skipped stocks won't affect the rest of the pipeline

  💾 FILE 1 saved → nifty50_for_powerbi.csv (12,151

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}


  ✅ [27/50] TATAMOTORS.NS
  ✅ [28/50] NESTLEIND.NS
  ✅ [29/50] DRREDDY.NS
  ✅ [30/50] DIVISLAB.NS
  ✅ [31/50] CIPLA.NS
  ✅ [32/50] COALINDIA.NS
  ✅ [33/50] BPCL.NS
  ✅ [34/50] GRASIM.NS
  ✅ [35/50] ADANIENT.NS
  ✅ [36/50] ADANIPORTS.NS
  ✅ [37/50] BAJAJFINSV.NS
  ✅ [38/50] BAJAJ-AUTO.NS
  ✅ [39/50] HEROMOTOCO.NS
  ✅ [40/50] EICHERMOT.NS
  ✅ [41/50] BRITANNIA.NS
  ✅ [42/50] HINDALCO.NS
  ✅ [43/50] UPL.NS
  ✅ [44/50] SBILIFE.NS
  ✅ [45/50] HDFCLIFE.NS
  ✅ [46/50] APOLLOHOSP.NS
  ✅ [47/50] TATACONSUM.NS
  ✅ [48/50] INDUSINDBK.NS
  ✅ [49/50] M&M.NS
  ✅ [50/50] LTF.NS

  💾 FILE 5 saved → nifty50_fundamentals_powerbi.csv (50 rows)

✅ [3/6] Complete

💬 [4/6] Fetching news sentiment...

  ✅ [01/50] RELIANCE.NS — Neutral
  ✅ [02/50] TCS.NS — Neutral
  ✅ [03/50] HDFCBANK.NS — Positive
  ✅ [04/50] INFY.NS — Neutral
  ✅ [05/50] ICICIBANK.NS — Positive
  ✅ [06/50] HINDUNILVR.NS — Positive
  ✅ [07/50] ITC.NS — Neutral
  ✅ [08/50] SBIN.NS — Positive
  ✅ [09/50] BHARTIARTL.NS — Positive
  ✅ [10/50] KO

In [6]:
# ════════════════════════════════════════════════════
# STANDALONE PORTFOLIO FIX
# Run this as a SEPARATE cell if files are still empty
# Does not depend on any previous variable
# ════════════════════════════════════════════════════

from pypfopt import EfficientFrontier, risk_models, expected_returns, DiscreteAllocation
import pandas as pd
import numpy as np
import os
import traceback

BASE = '/content/drive/MyDrive/indian_stock_analysis'

print("🔄 Loading data fresh from parquet...\n")

# ── Step 1: Load from saved parquet ──
try:
    tech_df = pd.read_parquet(
        f"{BASE}/data/processed/nifty50_technical.parquet"
    )
    tech_df["Date"] = pd.to_datetime(tech_df["Date"])
    if tech_df["Date"].dt.tz is not None:
        tech_df["Date"] = tech_df["Date"].dt.tz_localize(None)
    print(f"✅ Loaded nifty50_technical.parquet")
    print(f"   Shape: {tech_df.shape}")
    print(f"   Stocks: {tech_df['Ticker'].nunique()}")
    print(f"   Date range: {tech_df['Date'].min().date()} → "
          f"{tech_df['Date'].max().date()}")
except Exception as e:
    print(f"❌ Failed to load parquet: {e}")
    print("   Make sure Notebook 2 ran successfully first")
    raise

# ── Step 2: Build price matrix ──
print("\n📊 Building price matrix...")
prices = tech_df.pivot_table(
    index="Date", columns="Ticker", values="Close"
).sort_index()

if hasattr(prices.index, "tz") and prices.index.tz is not None:
    prices.index = prices.index.tz_localize(None)

prices = prices.dropna(thresh=int(len(prices) * 0.9), axis=1)
prices = prices.ffill().bfill()
prices = prices.loc[:, prices.notna().sum() >= 100]

print(f"✅ Price matrix: {prices.shape[1]} stocks × {len(prices)} days")
print(f"   Stocks: {list(prices.columns[:5])}... and more")

returns_daily  = prices.pct_change().dropna()
market_returns = returns_daily.mean(axis=1)
market_var     = market_returns.var()
trading_days   = 252
risk_free      = 0.065

# ── Step 3: Risk Metrics ──
print("\n📈 Computing risk metrics...")
risk_rows = []

for ticker in returns_daily.columns:
    try:
        r, mkt = returns_daily[ticker].dropna().align(
            market_returns, join="inner"
        )
        r = r.dropna()
        if len(r) < 30:
            continue

        ann_return   = round((1 + r.mean()) ** trading_days - 1, 4)
        ann_vol      = round(r.std() * np.sqrt(trading_days), 4)
        sharpe       = round(
            (ann_return - risk_free) / ann_vol, 3
        ) if ann_vol > 0 else 0.0

        cumulative   = (1 + r).cumprod()
        max_drawdown = round(
            ((cumulative - cumulative.cummax()) /
             cumulative.cummax()).min() * 100, 2
        )

        threshold = np.percentile(r, 5)
        var_95    = round(threshold * 100, 3)
        tail      = r[r <= threshold]
        cvar_95   = round(tail.mean() * 100, 3) if len(tail) > 0 else var_95

        cov_m = np.cov(r.values, mkt.values)
        beta  = round(cov_m[0][1] / market_var, 3) if market_var > 0 else 1.0

        curr_price = float(prices[ticker].iloc[-1])
        high_52w   = float(prices[ticker].tail(252).max())
        low_52w    = float(prices[ticker].tail(252).min())

        risk_rows.append({
            "Ticker":             ticker,
            "Ann_Return_Pct":     round(ann_return * 100, 2),
            "Ann_Volatility_Pct": round(ann_vol * 100, 2),
            "Sharpe_Ratio":       sharpe,
            "Max_Drawdown_Pct":   max_drawdown,
            "VaR_95_Pct":         var_95,
            "CVaR_95_Pct":        cvar_95,
            "Beta":               beta,
            "Current_Price":      round(curr_price, 2),
            "52W_High":           round(high_52w, 2),
            "52W_Low":            round(low_52w, 2),
            "From_52W_High_Pct":  round(
                (curr_price - high_52w) / high_52w * 100, 2
            )
        })

    except Exception as e:
        print(f"  ⚠️  {ticker} skipped — {e}")

risk_df = pd.DataFrame(risk_rows).sort_values(
    "Sharpe_Ratio", ascending=False
).reset_index(drop=True)

risk_df.to_parquet(
    f"{BASE}/data/processed/nifty50_risk_metrics.parquet", index=False
)
risk_df.to_csv(
    f"{BASE}/data/output/nifty50_risk_metrics_powerbi.csv", index=False
)

size = os.path.getsize(
    f"{BASE}/data/output/nifty50_risk_metrics_powerbi.csv"
) / 1024
print(f"✅ Risk metrics saved — {len(risk_df)} stocks | {size:.1f} KB")
print(f"\n   Top 5 by Sharpe Ratio:")
print(risk_df[["Ticker","Sharpe_Ratio","Ann_Return_Pct",
               "Ann_Volatility_Pct"]].head().to_string(index=False))

# ── Step 4: Portfolio Optimization ──
print("\n⚙️  Running portfolio optimization...")

try:
    mu = expected_returns.mean_historical_return(prices, frequency=252)
    S  = risk_models.CovarianceShrinkage(prices).ledoit_wolf()
    print(f"✅ Expected returns range: "
          f"{mu.min()*100:.1f}% to {mu.max()*100:.1f}%")

    perf_rows   = []
    best_weights = {}

    # ── Max Sharpe ──
    print("\n🔹 Max Sharpe...")
    try:
        ef = EfficientFrontier(mu, S)
        ef.add_constraint(lambda w: w >= 0.0)
        ef.add_constraint(lambda w: w <= 0.15)
        ef.max_sharpe(risk_free_rate=0.065)
        w  = ef.clean_weights(cutoff=0.005)
        p  = ef.portfolio_performance(
            verbose=False, risk_free_rate=0.065
        )
        best_weights = {k: v for k, v in w.items() if v > 0.005}
        perf_rows.append({
            "Portfolio":    "Max Sharpe",
            "Ann_Return":   round(p[0]*100, 2),
            "Ann_Risk":     round(p[1]*100, 2),
            "Sharpe_Ratio": round(p[2], 3)
        })
        print(f"   ✅ Return={p[0]*100:.2f}% Risk={p[1]*100:.2f}% "
              f"Sharpe={p[2]:.3f} Stocks={len(best_weights)}")
        for t, wt in sorted(
            best_weights.items(), key=lambda x: x[1], reverse=True
        ):
            print(f"      {t:<22} {wt*100:.2f}%")

    except Exception as e:
        print(f"   ❌ Max Sharpe failed: {e}")
        traceback.print_exc()

    # ── Min Volatility ──
    print("\n🔹 Min Volatility...")
    try:
        ef2 = EfficientFrontier(mu, S)
        ef2.add_constraint(lambda w: w >= 0.0)
        ef2.add_constraint(lambda w: w <= 0.15)
        ef2.min_volatility()
        w2 = ef2.clean_weights(cutoff=0.005)
        p2 = ef2.portfolio_performance(
            verbose=False, risk_free_rate=0.065
        )
        perf_rows.append({
            "Portfolio":    "Min Volatility",
            "Ann_Return":   round(p2[0]*100, 2),
            "Ann_Risk":     round(p2[1]*100, 2),
            "Sharpe_Ratio": round(p2[2], 3)
        })
        print(f"   ✅ Return={p2[0]*100:.2f}% Risk={p2[1]*100:.2f}% "
              f"Sharpe={p2[2]:.3f}")
    except Exception as e:
        print(f"   ❌ Min Volatility failed: {e}")

    # ── Max Return ──
    print("\n🔹 Max Return...")
    try:
        target = float(mu.max() * 0.80)
        ef3    = EfficientFrontier(mu, S)
        ef3.add_constraint(lambda w: w >= 0.0)
        ef3.add_constraint(lambda w: w <= 0.15)
        ef3.efficient_return(target_return=target)
        w3 = ef3.clean_weights(cutoff=0.005)
        p3 = ef3.portfolio_performance(
            verbose=False, risk_free_rate=0.065
        )
        perf_rows.append({
            "Portfolio":    "Max Return",
            "Ann_Return":   round(p3[0]*100, 2),
            "Ann_Risk":     round(p3[1]*100, 2),
            "Sharpe_Ratio": round(p3[2], 3)
        })
        print(f"   ✅ Return={p3[0]*100:.2f}% Risk={p3[1]*100:.2f}% "
              f"Sharpe={p3[2]:.3f}")
    except Exception as e:
        print(f"   ⚠️  Max Return failed ({e})")
        if perf_rows:
            fb = perf_rows[0].copy()
            fb["Portfolio"] = "Max Return"
            perf_rows.append(fb)
            print(f"   💡 Using Max Sharpe as fallback")

    # ── Save portfolio performance ──
    if perf_rows:
        perf_df = pd.DataFrame(perf_rows)
        perf_df.to_csv(
            f"{BASE}/data/output/portfolio_performance_powerbi.csv",
            index=False
        )
        size = os.path.getsize(
            f"{BASE}/data/output/portfolio_performance_powerbi.csv"
        ) / 1024
        print(f"\n✅ portfolio_performance_powerbi.csv "
              f"saved — {len(perf_df)} rows | {size:.1f} KB")
        print(perf_df.to_string(index=False))
    else:
        print("❌ perf_rows is empty — no performance data saved")

    # ── Discrete Allocation ──
    print("\n💰 Running discrete allocation...")

    if not best_weights:
        print("❌ best_weights is empty — cannot allocate")
    else:
        # Get prices for selected stocks only
        latest_prices_dict = {}
        for ticker in best_weights:
            if ticker in prices.columns:
                p_val = float(prices[ticker].iloc[-1])
                if p_val > 0:
                    latest_prices_dict[ticker] = p_val

        print(f"   Stocks selected: {len(latest_prices_dict)}")
        print(f"   Price range: ₹{min(latest_prices_dict.values()):,.0f} "
              f"— ₹{max(latest_prices_dict.values()):,.0f}")

        # Normalize weights
        w_valid = {
            k: v for k, v in best_weights.items()
            if k in latest_prices_dict
        }
        total_w      = sum(w_valid.values())
        weights_norm = {k: v/total_w for k, v in w_valid.items()}

        # Calculate minimum investment needed
        min_needed = sum(
            latest_prices_dict[t] * 1
            for t in weights_norm
        )
        INVESTMENT = max(100000, int(min_needed * 3 / 10000 + 1) * 10000)
        print(f"   Investment amount: ₹{INVESTMENT:,}")

        # Try allocation
        allocation = {}
        leftover   = INVESTMENT

        for method_name, method_fn in [
            ("greedy_portfolio", "greedy"),
            ("lp_portfolio",     "lp")
        ]:
            try:
                da = DiscreteAllocation(
                    weights_norm,
                    pd.Series(latest_prices_dict),
                    total_portfolio_value=INVESTMENT
                )
                if method_fn == "greedy":
                    allocation, leftover = da.greedy_portfolio()
                else:
                    allocation, leftover = da.lp_portfolio()
                print(f"   ✅ {method_name} succeeded")
                break
            except Exception as e:
                print(f"   ⚠️  {method_name} failed: {e}")
                continue

        # Build and save allocation dataframe
        if allocation:
            alloc_rows = []
            total_inv  = 0

            print(f"\n   {'Stock':<22} {'Shares':>7} "
                  f"{'Price':>12} {'Value':>12} {'Wt%':>7}")
            print(f"   {'─'*62}")

            for ticker, shares in sorted(
                allocation.items(),
                key=lambda x: x[1] * latest_prices_dict.get(x[0], 0),
                reverse=True
            ):
                if shares <= 0 or ticker not in latest_prices_dict:
                    continue
                price    = latest_prices_dict[ticker]
                value    = round(shares * price, 2)
                w_pct    = round(weights_norm.get(ticker, 0) * 100, 2)
                total_inv += value

                print(f"   {ticker.replace('.NS',''):<22} {shares:>7} "
                      f"₹{price:>10,.2f} ₹{value:>10,.2f} {w_pct:>6.1f}%")

                alloc_rows.append({
                    "Ticker":     ticker,
                    "Company":    ticker.replace(".NS", ""),
                    "Shares":     shares,
                    "Price":      round(price, 2),
                    "Value_INR":  value,
                    "Weight_Pct": w_pct
                })

            print(f"   {'─'*62}")
            print(f"   {'Total':<22} {'':>7} "
                  f"{'':>12} ₹{total_inv:>10,.2f}")
            print(f"   {'Leftover':<22} {'':>7} "
                  f"{'':>12} ₹{leftover:>10,.2f}")
            print(f"   {'Utilization':<22} {'':>7} "
                  f"{'':>12} "
                  f"{round(total_inv/INVESTMENT*100,1):>9.1f}%")

            alloc_df = pd.DataFrame(alloc_rows).sort_values(
                "Value_INR", ascending=False
            ).reset_index(drop=True)

            alloc_df.to_csv(
                f"{BASE}/data/output/portfolio_allocation_powerbi.csv",
                index=False
            )
            size = os.path.getsize(
                f"{BASE}/data/output/"
                f"portfolio_allocation_powerbi.csv"
            ) / 1024
            print(f"\n✅ portfolio_allocation_powerbi.csv "
                  f"saved — {len(alloc_df)} stocks | {size:.1f} KB")

        else:
            print("❌ allocation is empty after trying both methods")
            print("💡 Manually creating allocation from weights...")

            # Last resort — create allocation from weights directly
            # without using DiscreteAllocation
            INVESTMENT  = 500000  # increase to 5 lakh as last resort
            alloc_rows  = []
            total_inv   = 0

            for ticker, weight in sorted(
                weights_norm.items(),
                key=lambda x: x[1], reverse=True
            ):
                if ticker not in latest_prices_dict:
                    continue
                price       = latest_prices_dict[ticker]
                budget      = INVESTMENT * weight
                shares      = max(1, int(budget / price))
                value       = round(shares * price, 2)
                total_inv  += value

                alloc_rows.append({
                    "Ticker":     ticker,
                    "Company":    ticker.replace(".NS",""),
                    "Shares":     shares,
                    "Price":      round(price, 2),
                    "Value_INR":  value,
                    "Weight_Pct": round(weight * 100, 2)
                })
                print(f"   {ticker:<22} {shares} shares × "
                      f"₹{price:,.0f} = ₹{value:,.0f}")

            alloc_df = pd.DataFrame(alloc_rows).sort_values(
                "Value_INR", ascending=False
            )
            alloc_df.to_csv(
                f"{BASE}/data/output/portfolio_allocation_powerbi.csv",
                index=False
            )
            size = os.path.getsize(
                f"{BASE}/data/output/"
                f"portfolio_allocation_powerbi.csv"
            ) / 1024
            print(f"\n✅ portfolio_allocation_powerbi.csv "
                  f"saved (manual method) — "
                  f"{len(alloc_df)} stocks | {size:.1f} KB")

except Exception as e:
    print(f"\n❌ Portfolio section crashed completely: {e}")
    traceback.print_exc()

# ── Final verification ──
print("\n" + "=" * 55)
print("  PORTFOLIO FILES VERIFICATION")
print("=" * 55)

portfolio_files = [
    "nifty50_risk_metrics_powerbi.csv",
    "portfolio_performance_powerbi.csv",
    "portfolio_allocation_powerbi.csv"
]

for fname in portfolio_files:
    fpath = f"{BASE}/data/output/{fname}"
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        df   = pd.read_csv(fpath)
        print(f"  ✅ {fname}")
        print(f"     Size: {size:.1f} KB | Rows: {len(df)}")
        print(f"     Columns: {list(df.columns)}")
        print(f"     Preview:")
        print(df.head(3).to_string(index=False))
        print()
    else:
        print(f"  ❌ {fname} — NOT FOUND")

🔄 Loading data fresh from parquet...

✅ Loaded nifty50_technical.parquet
   Shape: (12151, 34)
   Stocks: 49
   Date range: 2025-06-30 → 2026-06-29

📊 Building price matrix...
✅ Price matrix: 49 stocks × 248 days
   Stocks: ['ADANIENT.NS', 'ADANIPORTS.NS', 'APOLLOHOSP.NS', 'ASIANPAINT.NS', 'AXISBANK.NS']... and more

📈 Computing risk metrics...
✅ Risk metrics saved — 49 stocks | 4.2 KB

   Top 5 by Sharpe Ratio:
      Ticker  Sharpe_Ratio  Ann_Return_Pct  Ann_Volatility_Pct
      LTF.NS         1.524           57.86               33.70
 HINDALCO.NS         1.465           47.09               27.71
EICHERMOT.NS         1.226           38.68               26.25
     SBIN.NS         1.202           33.40               22.38
 JSWSTEEL.NS         0.848           26.30               23.35

⚙️  Running portfolio optimization...
✅ Expected returns range: -38.2% to 49.3%

🔹 Max Sharpe...
   ✅ Return=30.81% Risk=14.07% Sharpe=1.728 Stocks=8
      COALINDIA.NS           15.00%
      EICHERMOT.NS 

In [7]:
import pandas as pd
BASE = '/content/drive/MyDrive/indian_stock_analysis'

# Reload and fix portfolio performance
perf_df = pd.read_csv(
    f"{BASE}/data/output/portfolio_performance_powerbi.csv"
)

# Add a ranking column for Power BI sorting
perf_df["Rank"] = [1, 3, 2]  # Max Sharpe best, Min Vol conservative

# Add a description column
perf_df["Description"] = [
    "Best risk-adjusted return",
    "Lowest possible risk",
    "Highest expected return"
]

# Add color coding for Power BI
perf_df["Color"] = ["#2CA02C", "#1F77B4", "#FF7F0E"]

perf_df.to_csv(
    f"{BASE}/data/output/portfolio_performance_powerbi.csv",
    index=False
)

print("✅ Portfolio performance updated!")
print(perf_df.to_string(index=False))


✅ Portfolio performance updated!
     Portfolio  Ann_Return  Ann_Risk  Sharpe_Ratio  Rank               Description   Color
    Max Sharpe       30.81     14.07         1.728     1 Best risk-adjusted return #2CA02C
Min Volatility       -3.70      8.84        -1.154     3      Lowest possible risk #1F77B4
    Max Return       30.81     14.07         1.728     2   Highest expected return #FF7F0E
